# Project 8 — Module 9: Fundamentos de Big Data
## Lesson 02 — Data Understanding

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 02 — Data Understanding |
| **Module** | M9 — Fundamentos de Big Data (Alkemy Bootcamp) |
| **Dataset** | Brazilian E-Commerce (Olist) + Synthetic clickstream |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> This notebook configures the PySpark environment (SparkSession + Java 17 Temurin),
> loads all 9 Olist CSV tables as RDDs and DataFrames, validates connectivity with
> basic actions (`count`, `take`, `printSchema`), generates synthetic clickstream
> data using `spark.range()`, and produces an initial data profile. The key output
> is a verified Spark environment with all data sources loaded and validated.

## Table of Contents

1. [Environment Setup — PySpark Configuration](#1-environment-setup--pyspark-configuration)
2. [Load Olist Data — CSV to RDDs](#2-load-olist-data--csv-to-rdds)
3. [Basic Actions — Validate Connectivity](#3-basic-actions--validate-connectivity)
4. [Load Olist Data — CSV to DataFrames](#4-load-olist-data--csv-to-dataframes)
5. [Schema Inspection and Data Profiling](#5-schema-inspection-and-data-profiling)
6. [Generate Synthetic Clickstream Data](#6-generate-synthetic-clickstream-data)
7. [Data Sources Summary](#7-data-sources-summary)
8. [LEAN Filter — Waste Elimination Review](#8-lean-filter--waste-elimination-review)
9. [Decisions Log — Lesson 02](#9-decisions-log--lesson-02)
10. [Next Steps — Lesson 03 Preview](#10-next-steps--lesson-03-preview)

---

## 1. Environment Setup — PySpark Configuration

Configure `PYSPARK_PYTHON` before any PySpark import to avoid subprocess errors.
SparkSession is the unified entry point for Spark 2.x+ (replaces separate
SparkContext + SQLContext).

In [ ]:
# =============================================================================
# IMPORTS
# Standard Library
import os
import sys
import warnings
from pathlib import Path

# ===== PySpark Python path (MUST be before PySpark imports) =====
os.environ['PYSPARK_PYTHON'] = sys.executable

# Core Data Science
import numpy as np
import pandas as pd

# Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, TimestampType
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

# ===== Plot Style =====
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')

# ===== Paths =====
DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_FINAL     = Path('../data/final')

print('Imports ready.')
print(f'Python: {sys.version}')
print(f'Data path: {DATA_RAW}')

In [ ]:
# =============================================================================
# SPARK SESSION
# =============================================================================
spark = (
    SparkSession.builder
    .appName('RetailMax-BigData-Pipeline')
    .master('local[*]')  # Use all available cores
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')  # Optimize for local mode
    .config('spark.ui.showConsoleProgress', 'false')  # Clean notebook output
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel('WARN')  # Reduce verbose INFO logs

print(f'Spark version : {spark.version}')
print(f'App name      : {spark.sparkContext.appName}')
print(f'Master        : {spark.sparkContext.master}')
print(f'Driver memory : 4g')
print(f'Java version  : {sc._jvm.System.getProperty("java.version")}')
print('\nSparkSession ready.')

---

## 2. Load Olist Data — CSV to RDDs

Load raw CSV files as RDDs using `sc.textFile()`. This demonstrates the
low-level Spark API (consigna L2 requirement). We skip the header row
manually since RDDs have no schema awareness.

In [ ]:
# =============================================================================
# LOAD CSV FILES AS RDDs
# =============================================================================
# Define file paths
olist_files = {
    'orders':       DATA_RAW / 'olist_orders_dataset.csv',
    'order_items':  DATA_RAW / 'olist_order_items_dataset.csv',
    'payments':     DATA_RAW / 'olist_order_payments_dataset.csv',
    'reviews':      DATA_RAW / 'olist_order_reviews_dataset.csv',
    'products':     DATA_RAW / 'olist_products_dataset.csv',
    'customers':    DATA_RAW / 'olist_customers_dataset.csv',
    'sellers':      DATA_RAW / 'olist_sellers_dataset.csv',
    'geolocation':  DATA_RAW / 'olist_geolocation_dataset.csv',
    'categories':   DATA_RAW / 'product_category_name_translation.csv',
}

# Load each file as RDD (text lines)
rdds_raw = {}
for name, path in olist_files.items():
    rdds_raw[name] = sc.textFile(str(path))

print(f'Loaded {len(rdds_raw)} RDDs from CSV files.')

---

## 3. Basic Actions — Validate Connectivity

Execute basic RDD actions to confirm data is loaded correctly.
Actions trigger actual computation (lazy evaluation ends here).

In [ ]:
# =============================================================================
# RDD ACTIONS: count() and take()
# =============================================================================
print('RDD Record Counts (including header row):')
print('-' * 45)

rdd_counts = {}
for name, rdd in rdds_raw.items():
    count = rdd.count()
    rdd_counts[name] = count
    print(f'  {name:20s} : {count:>10,} lines')

total = sum(rdd_counts.values())
print('-' * 45)
print(f'  {"TOTAL":20s} : {total:>10,} lines')

In [ ]:
# =============================================================================
# RDD SAMPLE: take() — first 3 rows of orders
# =============================================================================
print('Sample rows from orders RDD (raw text lines):')
print('=' * 80)
for i, row in enumerate(rdds_raw['orders'].take(3)):
    label = 'HEADER' if i == 0 else f'ROW {i}'
    print(f'[{label}] {row[:120]}...' if len(row) > 120 else f'[{label}] {row}')

---

## 4. Load Olist Data — CSV to DataFrames

Reload the same CSVs as Spark DataFrames with `spark.read.csv()`. This provides
schema inference, column names, and SQL capabilities — the preferred API for
structured data analysis in Spark 2.x+.

Both RDD and DataFrame loads are required by the consigna to demonstrate
understanding of both abstractions.

In [ ]:
# =============================================================================
# LOAD CSV FILES AS DATAFRAMES
# =============================================================================
dfs = {}
for name, path in olist_files.items():
    dfs[name] = spark.read.csv(
        str(path),
        header=True,
        inferSchema=True,
        encoding='utf-8'
    )

print(f'Loaded {len(dfs)} DataFrames.')
print()

# Show record counts (without header, now real data rows)
print('DataFrame Record Counts:')
print('-' * 45)
df_counts = {}
for name, df in dfs.items():
    count = df.count()
    df_counts[name] = count
    print(f'  {name:20s} : {count:>10,} rows')

total_rows = sum(df_counts.values())
print('-' * 45)
print(f'  {"TOTAL":20s} : {total_rows:>10,} rows')

---

## 5. Schema Inspection and Data Profiling

Inspect schemas and basic statistics for the core tables. This is the
Data Understanding phase — we need to know what we have before transforming.

In [ ]:
# =============================================================================
# SCHEMA INSPECTION — Core tables
# =============================================================================
core_tables = ['orders', 'order_items', 'payments', 'reviews', 'customers']

for table_name in core_tables:
    print(f'\n{"=" * 60}')
    print(f'Schema: {table_name}')
    print(f'{"=" * 60}')
    dfs[table_name].printSchema()

In [ ]:
# =============================================================================
# SAMPLE ROWS — orders table
# =============================================================================
print('Sample: orders (first 5 rows)')
dfs['orders'].show(5, truncate=30)

In [ ]:
# =============================================================================
# DATA PROFILING — Null counts per core table
# =============================================================================
print('Null Counts per Column (core tables):')
print()

for table_name in core_tables:
    df = dfs[table_name]
    total = df.count()
    print(f'--- {table_name} ({total:,} rows) ---')

    null_counts = df.select(
        [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
         for c in df.columns]
    ).collect()[0]

    for col_name in df.columns:
        nulls = null_counts[col_name]
        pct = (nulls / total) * 100 if total > 0 else 0
        flag = ' ⚠️' if pct > 5 else ''
        print(f'  {col_name:40s} : {nulls:>8,} nulls ({pct:5.1f}%){flag}')
    print()

---

## 6. Generate Synthetic Clickstream Data

Generate browsing behavior data using `spark.range()` + distributed column
functions. This covers the 3rd data source required by the consigna
(navigation behavior) and demonstrates Spark-native data generation —
never Python loops + `createDataFrame` (driver-side antipattern).

In [ ]:
# =============================================================================
# GENERATE SYNTHETIC CLICKSTREAM — distributed with spark.range()
# =============================================================================
NUM_EVENTS = 500_000

# Get real customer IDs from Olist to link clickstream
customer_ids = (
    dfs['customers']
    .select('customer_id')
    .distinct()
)
n_customers = customer_ids.count()
print(f'Unique customers available: {n_customers:,}')

# Generate clickstream events
clickstream = (
    spark.range(0, NUM_EVENTS)
    .withColumn('session_id',
        F.concat(F.lit('sess_'), F.col('id').cast('string')))
    .withColumn('customer_idx',
        (F.col('id') % n_customers).cast('int'))
    .withColumn('page_viewed',
        F.array(
            F.lit('home'), F.lit('product'), F.lit('category'),
            F.lit('cart'), F.lit('checkout')
        ).getItem((F.col('id') % 5).cast('int')))
    .withColumn('time_spent_seconds',
        (F.abs(F.hash(F.col('id'))) % 296 + 5).cast('int'))
    .withColumn('device',
        F.array(
            F.lit('desktop'), F.lit('mobile'), F.lit('tablet')
        ).getItem((F.col('id') % 3).cast('int')))
    .withColumn('event_timestamp',
        F.date_add(
            F.lit('2017-01-01').cast('date'),
            (F.col('id') % 730).cast('int')  # ~2 years range
        ).cast('timestamp'))
    .drop('id', 'customer_idx')
)

print(f'Clickstream events generated: {clickstream.count():,}')
print()
clickstream.printSchema()
clickstream.show(5, truncate=False)

In [ ]:
# =============================================================================
# LINK CLICKSTREAM TO REAL CUSTOMER IDs
# =============================================================================
# Add row number to customers for joining
from pyspark.sql.window import Window

customers_indexed = (
    customer_ids
    .withColumn('cust_idx',
        F.row_number().over(Window.orderBy('customer_id')) - 1)
)

# Recreate clickstream with index for join
clickstream_with_idx = (
    spark.range(0, NUM_EVENTS)
    .withColumn('session_id',
        F.concat(F.lit('sess_'), F.col('id').cast('string')))
    .withColumn('cust_idx',
        (F.col('id') % n_customers).cast('int'))
    .withColumn('page_viewed',
        F.array(
            F.lit('home'), F.lit('product'), F.lit('category'),
            F.lit('cart'), F.lit('checkout')
        ).getItem((F.col('id') % 5).cast('int')))
    .withColumn('time_spent_seconds',
        (F.abs(F.hash(F.col('id'))) % 296 + 5).cast('int'))
    .withColumn('device',
        F.array(
            F.lit('desktop'), F.lit('mobile'), F.lit('tablet')
        ).getItem((F.col('id') % 3).cast('int')))
    .withColumn('event_timestamp',
        F.date_add(
            F.lit('2017-01-01').cast('date'),
            (F.col('id') % 730).cast('int')
        ).cast('timestamp'))
    .drop('id')
)

# Join to get real customer_id
clickstream_final = (
    clickstream_with_idx
    .join(customers_indexed, on='cust_idx', how='inner')
    .select('session_id', 'customer_id', 'page_viewed',
            'time_spent_seconds', 'device', 'event_timestamp')
)

# Add to dfs dictionary
dfs['clickstream'] = clickstream_final

print(f'Clickstream with real customer IDs: {clickstream_final.count():,} events')
clickstream_final.show(5, truncate=False)

---

## 7. Data Sources Summary

In [ ]:
# =============================================================================
# DATA SOURCES SUMMARY
# =============================================================================
print('RetailMax Data Sources — Final Summary')
print('=' * 65)
print(f'{"Source":20s} {"Rows":>12s} {"Columns":>8s} {"Type":>15s}')
print('-' * 65)

grand_total = 0
for name, df in dfs.items():
    rows = df.count()
    cols = len(df.columns)
    dtype = 'Synthetic' if name == 'clickstream' else 'Real (Olist)'
    print(f'  {name:20s} {rows:>10,} {cols:>8d}   {dtype}')
    grand_total += rows

print('-' * 65)
print(f'  {"GRAND TOTAL":20s} {grand_total:>10,}')
print()
print(f'Total data sources: {len(dfs)}')
print(f'Real sources: {len(dfs) - 1} (Olist) | Synthetic: 1 (clickstream)')

In [ ]:
# =============================================================================
# STOP SPARK SESSION
# Uncomment only when done with all exploration in this notebook.
# Keep alive if continuing to next cells.
# =============================================================================
# spark.stop()
# print('SparkSession stopped.')

---

## 8. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---|---|---|
| Does every analysis link to a business decision? | ✅ All loads and profiles feed into the transformation + modeling phases | Proceed |
| Is there redundancy between sections? | ✅ RDD load demonstrates low-level API, DataFrame load demonstrates high-level API — both required by consigna | Proceed |
| Could we skip the RDD load? | ⚠️ Technically yes (DataFrames are superior), but L2 consigna explicitly requires RDD actions | Keep — consigna compliance |
| Is the clickstream volume appropriate? | ✅ 500K events is large enough to demonstrate Spark value, small enough for local processing | Proceed |

---

## 9. Decisions Log — Lesson 02

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|----------|-----------|------------------------|-------------|
| D1 | `local[*]` master with 4g driver memory | Uses all available CPU cores; 4g handles ~120 MB CSV + 500K synthetic without OOM | `local[2]` (underutilizes CPU), `8g` (excessive for this data volume) | ✅ |
| D2 | `spark.sql.shuffle.partitions = 8` | Default 200 is excessive for local mode with ~2M rows; 8 reduces task overhead | Default 200 (too many small tasks), 4 (too few for multi-core) | ✅ |
| D3 | Load data both as RDDs and DataFrames | Consigna L2 requires RDD actions; DataFrames needed for SQL + MLlib in later notebooks | RDD-only (loses SQL capability), DataFrame-only (misses consigna requirement) | ✅ |
| D4 | `inferSchema=True` for initial DataFrame load | Adequate for data understanding phase; explicit schemas will be applied in NB 03 (Data Preparation) | Manual schema definition now (premature — we haven't explored the data yet) | ✅ |
| D5 | Generate 500K clickstream events | Large enough for Spark to add value vs. pandas; small enough for local processing in < 1 min | 100K (too small), 2M (slow on local for development) | ✅ |
| D6 | Link clickstream to real Olist customer_ids | Enables joins with transaction data in NB 04; maintains referential integrity | Random UUIDs (no join capability), subset of customers (incomplete coverage) | ✅ |

---

## 10. Next Steps — Lesson 03 Preview

| Priority | Next Step | Target |
|---|---|---|
| 🔴 Immediate | Create RDDs and Pair RDDs from transactions — apply map, filter, flatMap, distinct, sortBy | NB 03 |
| 🔴 Immediate | Execute actions (collect on small sets, sum, mean, stdev) and document linaje | NB 03 |
| 🟡 Short-term | Document the DAG generated by transformation chains | NB 03 |
| 🟡 Short-term | Apply cache/persist strategy on reused RDDs | NB 03 |
| 🟢 Long-term | Transform RDDs to DataFrames with explicit schemas for Spark SQL (NB 04) | NB 04 |

---

**← Previous:** [01 — Business Understanding](./01_business_understanding.ipynb)
**Next Phase →** [03 — Data Preparation](./03_data_preparation.ipynb)